# Encoding categorical variables: one-hot, dummy, and effect coding

_Feature Engineering_

**Turn a category like city or browser into numbers a model can use, without inventing a fake order.**

A categorical variable is a column whose values are names, not numbers:
       a city (SF, NYC, Seattle), a browser
       (Chrome, Safari), a product category. The categories have no built-in
       order &mdash; NYC is not "bigger than" SF. But almost every model wants
       numbers as input. So we have to turn names into numbers without smuggling in a false
       ordering.

       The naive trick &mdash; just label the cities 0, 1, 2 &mdash; is exactly the mistake.
       It tells a linear model that Seattle (2) is twice NYC (1) and that the gap
       SF&rarr;NYC equals the gap NYC&rarr;Seattle. None of that is true. Chapter 5
       of Zheng & Casari's Feature Engineering for Machine Learning shows the right way: give each
       category its own column. The chapter's running example is a tiny table of cities and their
       rents.

---

This notebook is a practice scaffold for the **Encoding categorical variables: one-hot, dummy, and effect coding** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## See it for yourself: the problem, then the fix

Run this cell. It plots the **raw** data and reproduces the problem it causes, then plots the **engineered** data and shows the fix — with before/after numbers.

In [ ]:
# fe-categorical-encoding.py
# LESSON: One-hot encoding a nominal category vs naively label-encoding it as integers.
#
# PROBLEM we reproduce: a category like "city" has NO natural order. If we just hand
# the model integer codes 0,1,2,3, a LINEAR model reads those integers as NUMBERS:
# it assumes city 3 is "3x" city 1, and that city 1 sits "between" city 0 and city 2.
# That is a FALSE ORDER and FALSE DISTANCES. When the real city->target pattern is
# NON-monotonic in the arbitrary code (cities 0 and 2 high, cities 1 and 3 low), a
# single weight times the integer cannot fit it -> poor accuracy.
# FIX: one-hot encode. Each city gets its OWN 0/1 column, so the model learns one
# independent weight per city -> it can turn on exactly cities 0 and 2 -> accuracy jumps.
#
# Runs top-to-bottom in Colab. Packages: numpy, pandas, scikit-learn, matplotlib only.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# ---------------------------------------------------------------------------
# STEP 1: Build the data with a fixed rng (illustrative — a nominal category with
# a NON-monotonic relationship to the target).  4 cities, coded 0..3.
# We choose target probability per city so it is HIGH for 0 and 2, LOW for 1 and 3:
# i.e. it zig-zags in the integer code, which no single line can follow.
# ---------------------------------------------------------------------------
rng = np.random.default_rng(0)                  # fixed seed -> reproducible numbers
n = 800
city = rng.integers(0, 4, size=n)               # arbitrary integer code 0,1,2,3
p_by_city = np.array([0.85, 0.15, 0.85, 0.15])  # zig-zag: 0,2 high ; 1,3 low
y = (rng.uniform(size=n) < p_by_city[city]).astype(int)

df = pd.DataFrame({"city": city, "y": y})

# Split ONCE on the raw integer column; reuse the same rows for both encodings.
idx = np.arange(n)
idx_tr, idx_te = train_test_split(idx, test_size=0.3, random_state=0, stratify=y)

# ---------------------------------------------------------------------------
# STEP 2: Visualize the PURE (raw) data — target rate per city.
# This bar chart shows the relationship ZIG-ZAGS in the integer code (non-monotonic),
# which is exactly what a "city * single weight" linear term cannot capture.
# ---------------------------------------------------------------------------
rates = df.groupby("city")["y"].mean()
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].bar(rates.index, rates.values, color=["steelblue", "salmon", "steelblue", "salmon"])
ax[0].plot(rates.index, rates.values, "k--o", lw=1.5, label="rate vs integer code")
ax[0].set_title("STEP 2: target rate per city — ZIG-ZAGS in the integer code")
ax[0].set_xlabel("city (integer code 0..3)"); ax[0].set_ylabel("rate of y=1")
ax[0].set_xticks([0, 1, 2, 3]); ax[0].set_ylim(0, 1.05); ax[0].legend(loc="upper right")

# ---------------------------------------------------------------------------
# STEP 3: Reproduce the PROBLEM — LogisticRegression on the RAW integer code.
# The model sees ONE number per row and fits ONE weight: P(y=1) = sigmoid(w*city + b),
# a monotone curve in city. It cannot say "high at 0 and 2 but low at 1 and 3".
# ---------------------------------------------------------------------------
X_int = df[["city"]].to_numpy(dtype=float)      # raw integer codes as a 2D matrix
int_clf = LogisticRegression()
int_clf.fit(X_int[idx_tr], y[idx_tr])
int_acc = accuracy_score(y[idx_te], int_clf.predict(X_int[idx_te]))

# ---------------------------------------------------------------------------
# STEP 4: Apply the FIX — one-hot encode the city, then visualize the new columns.
# pandas.get_dummies turns the single "city" column into 4 separate 0/1 columns.
# ---------------------------------------------------------------------------
X_oh = pd.get_dummies(df["city"], prefix="city").to_numpy(dtype=float)

# Show the one-hot layout for one example row per city (the identity-like block).
sample = np.vstack([X_oh[df["city"].to_numpy() == c][0] for c in range(4)])
ax[1].imshow(sample, cmap="Blues", vmin=0, vmax=1, aspect="auto")
ax[1].set_title("STEP 4: one-hot columns — each city is its OWN 0/1 feature")
ax[1].set_xlabel("one-hot column"); ax[1].set_ylabel("example from city")
ax[1].set_xticks(range(4)); ax[1].set_xticklabels([f"city_{c}" for c in range(4)])
ax[1].set_yticks(range(4)); ax[1].set_yticklabels([f"city {c}" for c in range(4)])
for i in range(4):
    for j in range(4):
        ax[1].text(j, i, int(sample[i, j]), ha="center", va="center")
plt.tight_layout(); plt.show()

# ---------------------------------------------------------------------------
# STEP 5: Show the FIX — the SAME LogisticRegression on the one-hot features.
# Now there is one independent weight per city, so it can lift cities 0 and 2 and
# drop cities 1 and 3 -> accuracy jumps toward the ~0.85 best-possible (label noise).
# ---------------------------------------------------------------------------
oh_clf = LogisticRegression()
oh_clf.fit(X_oh[idx_tr], y[idx_tr])
oh_acc = accuracy_score(y[idx_te], oh_clf.predict(X_oh[idx_te]))

print(f"PROBLEM (label-encoded integers): accuracy = {int_acc:.3f}")
print(f"FIX (one-hot encoded):            accuracy = {oh_acc:.3f}")
print(f"PROBLEM (raw): {int_acc:.3f}   →   FIX (engineered): {oh_acc:.3f}")

## Reference implementation — pandas + scikit-learn

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression

# --- The book's tiny city -> rent table (Chapter 5) ---
df = pd.DataFrame({
    'city': ['SF', 'SF', 'NYC', 'NYC', 'Seattle', 'Seattle'],
    'rent': [4001, 3999, 3501, 3499, 2501, 2499],   # ~ means 4000 / 3500 / 2500
})

# === One-hot encoding ===  one binary column per city (k columns)
one_hot = pd.get_dummies(df['city'], prefix='city')
print(one_hot)
#    city_NYC  city_SF  city_Seattle
# 0         0        1             0   <- SF
# ...

# scikit-learn's OneHotEncoder (handles unseen categories at test time)
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
X_ohe = ohe.fit_transform(df[['city']])
print(ohe.categories_)        # [array(['NYC', 'SF', 'Seattle'], ...)]

# === Dummy coding ===  drop one column -> reference category is all-zeros (k-1 cols)
dummy = pd.get_dummies(df['city'], prefix='city', drop_first=True)
# 'NYC' is dropped -> NYC is the reference, coded (0, 0).
lin = LinearRegression().fit(dummy, df['rent'])
print('dummy intercept :', lin.intercept_)   # ~ NYC mean (the reference)
print('dummy coefs     :', lin.coef_)         # each = (city mean) - (NYC mean)

# === Effect coding ===  reference category coded as all -1 (k-1 cols)
effect = dummy.copy().astype(float)
ref_rows = (df['city'] == 'NYC').values       # the reference category
effect.loc[ref_rows, :] = -1.0                # all-zeros row -> all -1
lin2 = LinearRegression().fit(effect, df['rent'])
print('effect intercept:', lin2.intercept_)   # ~ GRAND MEAN of the city means
print('effect coefs    :', lin2.coef_)         # each = (city mean) - (grand mean)

## Visualize the data & results

_Bin a real wine feature ('color_intensity' from load_wine) into 3 named categories (pale / medium / deep). What does the one-hot matrix look like, and how many columns does one-hot use versus dummy coding?_

In [ ]:
import numpy as np
from sklearn.datasets import load_wine
from sklearn.preprocessing import OneHotEncoder

d = load_wine()                                  # 178 real wine samples
fi = list(d.feature_names).index('color_intensity')
ci = d.data[:, fi]                               # a continuous numeric feature

# --- Bin into 3 NAMED categories at the terciles (33rd / 67th percentile) ---
edges = np.percentile(ci, [0, 100/3, 200/3, 100])
names = np.array(['pale', 'medium', 'deep'])
idx = np.clip(np.digitize(ci, edges[1:-1]), 0, 2)
labels = names[idx]                              # now a CATEGORICAL column

# Category frequencies (the bar chart):
for c in names:
    print(c, int((labels == c).sum()))           # -> pale 60 / medium 58 / deep 60

# --- One-hot encode; take 3 sample rows per category for the heatmap ---
order = ['pale', 'medium', 'deep']
samp = np.concatenate([np.where(labels == c)[0][:3] for c in order])
ohe = OneHotEncoder(sparse_output=False, categories=[order])
M = ohe.fit_transform(labels[samp].reshape(-1, 1)).astype(int)
print(M)                                          # block-diagonal: one 1 per row
print('one-hot columns :', M.shape[1])            # -> 3   (k categories)
print('dummy columns   :', M.shape[1] - 1)        # -> 2   (drop the reference)

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** You have a browser column with values Chrome, Safari, Firefox, Edge and you feed it to a logistic regression that includes an intercept. A teammate one-hot encodes it into 4 columns and the fit warns about a singular matrix. What happened and what is the fix?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Notice the 4 one-hot columns always sum to 1 for every row. — _Exactly one browser is "hot", so the four binary columns add to a constant 1._
- See that the model's intercept is also a constant-1 column. — _Intercept = sum of the four one-hot columns is an exact linear dependence — the dummy-variable trap._
- Drop one column: dummy coding with drop_first=True (or effect coding). — _Removing one category as the reference breaks the dependence; the matrix becomes full rank and coefficients are unique._

**Answer:** Full one-hot columns sum to 1, which is collinear with the intercept &mdash; the
         dummy-variable trap, giving a singular (rank-deficient) design matrix. Switch to
         dummy coding (pd.get_dummies(..., drop_first=True)) or effect coding:
         3 columns with one browser as the reference. Alternatively, drop the intercept or add
         regularization, which also resolves the non-uniqueness.

</details>

**Problem 2.** Encode three cities (SF, NYC, Seattle, with mean rents 4000, 3500, 2500) using effect coding with Seattle as the reference, then read off what the linear model's intercept and SF coefficient mean.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Build the codes: SF=(1,0), NYC=(0,1), Seattle=(-1,-1). — _Effect coding uses $k-1=2$ columns and codes the reference category as all &minus;1._
- Recall that under effect coding the intercept equals the grand mean. — _Grand mean $\bar y=(4000+3500+2500)/3=3333.3$, so $\beta_0=3333.3$._
- Compute the SF coefficient as SF's deviation from the grand mean. — _$\beta_{\text{SF}}=4000-3333.3=666.7$; each coefficient is a deviation from the overall average._

**Answer:** Codes: $\text{SF}=(1,0)$, $\text{NYC}=(0,1)$, $\text{Seattle}=(-1,-1)$. The intercept is
         the grand mean $\beta_0=\bar{y}=3333.3$. The SF coefficient is SF's deviation from that
         average, $\beta_{\text{SF}}=4000-3333.3=666.7$. Seattle's deviation is recovered as
         $-(\beta_{\text{SF}}+\beta_{\text{NYC}})=-(666.7+166.7)=-833.3$, i.e. Seattle sits 833 below the
         grand mean.

</details>

**Problem 3.** You must encode a zip_code column with about 30,000 distinct values for a linear model. Why is plain one-hot encoding a poor choice here, and what does Chapter 5 suggest instead?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Count the columns one-hot would create: one per zip code. — _30,000 categories &rarr; 30,000 mostly-zero columns &mdash; the high-cardinality blow-up._
- Note the cost: huge sparse matrices, lots of memory, slow training, and many columns seen only a few times. — _Rare categories give unreliable weights and the dimensionality explodes._
- Reach for the chapter's high-cardinality tools. — _Feature hashing caps the column count by hashing categories into a fixed number of buckets; bin-counting replaces the category with target statistics._

**Answer:** One-hot makes one column per zip code &mdash; ~30,000 sparse columns &mdash; which is
         memory-hungry, slow, and full of rarely-seen features. This is the high-cardinality blow-up.
         Chapter 5's answer is feature hashing (hash categories into a fixed, small number of columns)
         or bin-counting (replace each category with statistics of the target, e.g. its click-through
         rate), both of which keep the dimensionality bounded.

</details>